In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_task_interference"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
NPZ_DIR = os.path.join(OUT_DIR, "npz")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Small enough for Kaggle/T4, but enough to show signal.
GSM8K_N = 5
STRATEGYQA_N = 5
MMLU_PER_SUBJECT = 3
MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
]

# Gradient analysis layers: last 6 layers by default.
GRAD_LAYER_COUNT = 6
# Representation analysis layers: all layers by default, but saved efficiently.
REP_LAYER_STRIDE = 1
# How many top PCs to use for subspace overlap.
SUBSPACE_K = 8

# Generation settings for evaluation
MAX_NEW_GSM8K = 64
MAX_NEW_STRATEGYQA = 16
MAX_NEW_MMLU = 16

# Prompt styles chosen to match the task-specific setups you found useful.
USE_GSM8K_REASONING_PROMPT = True
USE_STRATEGYQA_REASONING_PROMPT = False
USE_MMLU_DELIMITER = True

# If True, compare with the plain non-delimited MMLU prompt too.
RUN_MMLU_PLAIN_BASELINE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(NPZ_DIR, exist_ok=True)

TOKENIZER = None
MODEL = None
MODEL_NUM_LAYERS = None
MODEL_HIDDEN = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None

TASKS = ["gsm8k", "strategyqa", "mmlu"]

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def preview_text(s, max_len=220):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

def cosine(a, b, eps=1e-12):
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    b = np.asarray(b, dtype=np.float64).reshape(-1)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return float("nan")
    return float(np.dot(a, b) / (na * nb + eps))

# ============================================================
# ADVANCED PLOTTING UTILS
# ============================================================

def bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, color='#4C72B0')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def line_plot(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10, 5))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.6, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.55), max(5, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    for i in range(len(ylabels)):
        for j in range(len(xlabels)):
            val = mat[i, j]
            color = "white" if abs(val) < (np.nanmax(np.abs(mat)) / 2) else "black"
            if not np.isnan(val):
                plt.text(j, i, f"{val:.2f}", ha="center", va="center", color=color)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def histogram(values, title, path, xlabel="Value", bins=12):
    plt.figure(figsize=(8.8, 4.2))
    plt.hist(values, bins=bins, color='#DD8452', edgecolor='black', alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def scatter(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(6.8, 5.2))
    plt.scatter(x, y, alpha=0.8, color='#55A868')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def hexbin_plot(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(x, y, gridsize=15, cmap='inferno', mincnt=1)
    cb = plt.colorbar(hb)
    cb.set_label('Density')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def violin_layer_plot(df, val_col, layer_col, title, path, ylabel="Value"):
    layers = sorted(df[layer_col].unique())
    data = [df[df[layer_col] == l][val_col].dropna().values for l in layers]
    
    plt.figure(figsize=(10, 5))
    if any(len(d) > 0 for d in data):
        parts = plt.violinplot(data, positions=range(len(layers)), showmeans=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#55A868')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.title(title)
        plt.xlabel("Layer")
        plt.ylabel(ylabel)
        plt.xticks(range(len(layers)), layers)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def correlation_heatmap(df, cols, title, path):
    df_numeric = df.copy()
    for col in cols:
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
    
    sub_df = df_numeric[cols].dropna()
    if len(sub_df) < 2: return
    
    corr = sub_df.corr().values
    plt.figure(figsize=(8, 6))
    im = plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im)
    plt.xticks(range(len(cols)), cols, rotation=30, ha='right')
    plt.yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            color = 'black' if abs(corr[i, j]) < 0.5 else 'white'
            plt.text(j, i, f"{corr[i, j]:.2f}", ha='center', va='center', color=color)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def radar_chart(categories, values_dict, title, path):
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    
    plt.xticks(angles[:-1], categories, size=10)
    ax.set_rlabel_position(0)
    
    for label, values in values_dict.items():
        vals = list(values)
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, linestyle='solid', label=label)
        ax.fill(angles, vals, alpha=0.25)
        
    plt.title(title, size=15, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# MATH UTILS
# ============================================================

def center_rows(X):
    X = np.asarray(X, dtype=np.float64)
    return X - X.mean(axis=0, keepdims=True)

def top_pca_basis(X, k=8):
    Xc = center_rows(X)
    n, d = Xc.shape
    if n < 2 or d < 2:
        k_eff = min(k, d)
        U = np.eye(d, k_eff)
        return U.astype(np.float64), 0.0
    k_eff = min(k, d, max(1, n - 1))
    _, s, vt = np.linalg.svd(Xc, full_matrices=False)
    basis = vt[:k_eff].T
    explained = float(np.sum(s[:k_eff] ** 2) / (np.sum(s ** 2) + 1e-12))
    return basis.astype(np.float64), explained

def subspace_overlap(U, V):
    U = np.asarray(U, dtype=np.float64)
    V = np.asarray(V, dtype=np.float64)
    k = min(U.shape[1], V.shape[1])
    if k == 0:
        return float("nan")
    M = U.T @ V
    return float(np.linalg.norm(M, ord="fro") ** 2 / (k + 1e-12))

def linear_cka(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if X.shape[0] != Y.shape[0] or X.shape[0] < 2:
        return float("nan")
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    XT_Y = X.T @ Y
    hsic = np.linalg.norm(XT_Y, ord="fro") ** 2
    denom = math.sqrt(max(0, (np.linalg.norm(X.T @ X, ord="fro") ** 2) * (np.linalg.norm(Y.T @ Y, ord="fro") ** 2))) + 1e-12
    return float(hsic / denom)

def mean_pool_hidden(hidden, attention_mask=None):
    h = hidden[0].detach().float().cpu().numpy()
    if attention_mask is None:
        return h.mean(axis=0)
    mask = attention_mask[0].detach().float().cpu().numpy().reshape(-1, 1)
    denom = mask.sum(axis=0)
    if float(denom) <= 0:
        return h.mean(axis=0)
    return (h * mask).sum(axis=0) / (denom + 1e-12)

# ============================================================
# CACHE / SAMPLING
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# ANSWER / PROMPT HELPERS
# ============================================================

def extract_number(text):
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = re.sub(r"[$,]", "", span)
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", re.sub(r"[$,]", "", text))
    return nums[-1] if nums else None

def extract_yes_no(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text, choices=None):
    if text is None:
        return None
    span = text
    m = re.findall(r"<answer>(.*?)</answer>", span, flags=re.IGNORECASE | re.DOTALL)
    if m:
        span = m[-1]
    span_up = span.upper().strip()
    if span_up in ("A", "B", "C", "D"):
        return span_up
    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
        r"<ANSWER>\s*\(?([ABCD])\)?",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", span_up)
    if hits:
        return hits[-1].upper()
    if choices is not None:
        low = span.lower()
        for i, c in enumerate(choices):
            if str(c).strip().lower() in low:
                return chr(65 + i)
    return None

def same_numeric(a, b, tol=1e-6):
    try:
        na = float(re.sub(r"[$,]", "", str(a)))
        nb = float(re.sub(r"[$,]", "", str(b)))
        return abs(na - nb) <= tol
    except Exception:
        return False

def canonical_number_str(x):
    try:
        v = float(re.sub(r"[$,]", "", str(x)))
    except Exception:
        return None
    if abs(v - round(v)) < 1e-6:
        return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

def parse_prediction(task, text, choices=None):
    if task == "gsm8k":
        return canonical_number_str(extract_number(text))
    if task == "strategyqa":
        return extract_yes_no(text)
    if task == "mmlu":
        return extract_mcq(text, choices=choices)
    raise ValueError(task)

def exact_correct(task, pred, gold):
    if task == "gsm8k":
        return same_numeric(pred, gold)
    return pred == gold

def answer_letter_to_content(letter, choices):
    if letter is None:
        return None
    letter = letter.upper().strip()
    if letter not in "ABCD":
        return None
    idx = ord(letter) - 65
    if idx < 0 or idx >= len(choices):
        return None
    return choices[idx]

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def discover_blocks(model):
    candidates = [
        ("model.layers", getattr(getattr(model, "model", None), "layers", None)),
        ("model.decoder.layers", getattr(getattr(getattr(model, "model", None), "decoder", None), "layers", None)),
        ("transformer.h", getattr(getattr(model, "transformer", None), "h", None)),
        ("gpt_neox.layers", getattr(getattr(model, "gpt_neox", None), "layers", None)),
        ("decoder.layers", getattr(getattr(model, "decoder", None), "layers", None)),
    ]
    for name, maybe_blocks in candidates:
        if maybe_blocks is not None:
            maybe_blocks = list(maybe_blocks)
            if len(maybe_blocks) > 0:
                return maybe_blocks, name
    raise RuntimeError("Could not locate transformer blocks.")

def discover_final_norm(model):
    candidates = [
        ("model.norm", getattr(getattr(model, "model", None), "norm", None)),
        ("transformer.ln_f", getattr(getattr(model, "transformer", None), "ln_f", None)),
        ("gpt_neox.final_layer_norm", getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)),
        ("decoder.final_layer_norm", getattr(getattr(model, "decoder", None), "final_layer_norm", None)),
    ]
    for name, mod in candidates:
        if mod is not None:
            return mod, name
    return None, None

def discover_lm_head(model):
    if hasattr(model, "lm_head"):
        return model.lm_head, "lm_head"
    if hasattr(model, "embed_out"):
        return model.embed_out, "embed_out"
    raise RuntimeError("Could not locate lm_head / embed_out.")

def load_model():
    global MODEL, BLOCKS, FINAL_NORM, LM_HEAD, MODEL_NUM_LAYERS, MODEL_HIDDEN
    tok = load_tokenizer()

    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    blocks, block_name = discover_blocks(model)
    final_norm, norm_name = discover_final_norm(model)
    lm_head, head_name = discover_lm_head(model)

    MODEL = model
    BLOCKS = blocks
    FINAL_NORM = final_norm
    LM_HEAD = lm_head
    MODEL_NUM_LAYERS = len(BLOCKS)
    try:
        MODEL_HIDDEN = int(model.config.hidden_size)
    except Exception:
        MODEL_HIDDEN = None

    print(f"Loaded model with {MODEL_NUM_LAYERS} blocks ({block_name}), norm={norm_name}, head={head_name}")
    return model, tok

# ============================================================
# DATA LOADERS
# ============================================================

def load_gsm8k_samples(n=GSM8K_N):
    ds = load_dataset("gsm8k", "main", split="test")
    ds, idx = sample_dataset(ds, "gsm8k_task_interference", n)
    samples = []
    for i, s in enumerate(ds):
        ans = s["answer"]
        rationale, gold = ans.split("####", 1)
        gold = canonical_number_str(gold.strip())
        samples.append({
            "sample_id": i,
            "task": "gsm8k",
            "source_index": idx[i],
            "question": s["question"],
            "gold": gold,
            "completion": ans.strip(),
            "prompt": None,
        })
    return samples

def load_strategyqa_samples(n=STRATEGYQA_N):
    ds = load_dataset("ChilleD/StrategyQA", split="test")
    ds, idx = sample_dataset(ds, "strategyqa_task_interference", n)
    samples = []
    for i, s in enumerate(ds):
        gold = "yes" if bool(s["answer"]) else "no"
        completion = gold
        samples.append({
            "sample_id": i,
            "task": "strategyqa",
            "source_index": idx[i],
            "question": s["question"],
            "gold": gold,
            "completion": completion,
            "prompt": None,
        })
    return samples

def load_mmlu_samples(subjects=MMLU_SUBJECTS, n_per_subject=MMLU_PER_SUBJECT):
    all_samples = []
    for subject in subjects:
        ds = load_dataset("cais/mmlu", subject, split="test")
        ds, idx = sample_dataset(ds, f"mmlu_task_interference_{subject}", n_per_subject)
        for i, s in enumerate(ds):
            gold = chr(65 + int(s["answer"]))
            all_samples.append({
                "sample_id": len(all_samples),
                "task": "mmlu",
                "subject": subject,
                "source_index": idx[i],
                "question": s["question"],
                "choices": list(s["choices"]),
                "gold": gold,
                "completion": gold,
                "prompt": None,
            })
    return all_samples

# ============================================================
# PROMPTS
# ============================================================

def build_prompt(sample, variant=None):
    task = sample["task"]
    if task == "gsm8k":
        if USE_GSM8K_REASONING_PROMPT:
            return (
                f"Question: {sample['question']}\n"
                "Solve step by step.\n"
                "Answer:"
            )
        return f"Question: {sample['question']}\nAnswer:"

    if task == "strategyqa":
        if USE_STRATEGYQA_REASONING_PROMPT:
            return (
                f"Question: {sample['question']}\n"
                "Think carefully and answer with yes or no.\n"
                "Answer:"
            )
        return f"Question: {sample['question']}\nAnswer:"

    if task == "mmlu":
        choices = sample["choices"]
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
        if variant == "plain":
            header = "Question:"
        else:
            header = "## Question:" if USE_MMLU_DELIMITER else "Question:"
        return (
            f"{header} {sample['question']}\n"
            f"{opts}\n"
            "Answer:"
        )

    raise ValueError(task)

# ============================================================
# GENERATION / LOSS / GRADS
# ============================================================

@torch.inference_mode()
def greedy_generate(prompt, max_new_tokens=16):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=TOKENIZER.eos_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion

def build_teacher_forced_batch(sample, variant=None):
    prompt = build_prompt(sample, variant=variant)
    completion = sample["completion"]
    full_text = prompt + completion
    enc = TOKENIZER(full_text, return_tensors="pt")
    prompt_ids = TOKENIZER(prompt, return_tensors="pt").input_ids[0]
    labels = enc["input_ids"].clone()
    labels[:, : prompt_ids.shape[0]] = -100
    return prompt, completion, enc, labels, prompt_ids.shape[0]

def task_loss(sample, variant=None):
    prompt, completion, enc, labels, prompt_len = build_teacher_forced_batch(sample, variant=variant)
    with torch.inference_mode():
        out = MODEL(
            **{k: v.to(DEVICE) for k, v in enc.items()},
            labels=labels.to(DEVICE),
            use_cache=False,
            return_dict=True,
        )
    return float(out.loss.item()), prompt, completion, enc, labels, prompt_len

def capture_hook(layer_idx, storage):
    def hook(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        if torch.is_grad_enabled() and getattr(hidden, "requires_grad", False):
            hidden.retain_grad()
        storage[layer_idx] = hidden
        return out
    return hook

def collect_gradient_vectors(sample, grad_layers, variant=None):
    prompt, completion, enc, labels, prompt_len = build_teacher_forced_batch(sample, variant=variant)
    inputs = {k: v.to(DEVICE) for k, v in enc.items()}
    labels = labels.to(DEVICE)
    storage = {}
    hooks = []
    vecs = {}

    # Essential fix: Force gradient tracking by requiring gradients on embeddings
    embeds = MODEL.get_input_embeddings()
    orig_req_grad = embeds.weight.requires_grad
    embeds.weight.requires_grad_(True)

    try:
        for li in grad_layers:
            hooks.append(BLOCKS[li].register_forward_hook(capture_hook(li, storage)))

        MODEL.zero_grad(set_to_none=True)
        with torch.enable_grad():
            out = MODEL(
                **inputs,
                labels=labels,
                use_cache=False,
                output_hidden_states=True,
                return_dict=True,
            )
            out.loss.backward()

        for li in grad_layers:
            h = storage.get(li, None)
            if h is None or getattr(h, "grad", None) is None:
                continue
            grad = h.grad.detach().float().cpu().numpy()[0]
            if prompt_len < grad.shape[0]:
                comp_grad = grad[prompt_len:]
                if comp_grad.shape[0] == 0:
                    comp_grad = grad
            else:
                comp_grad = grad
            vec = comp_grad.mean(axis=0)
            vecs[li] = vec.astype(np.float32)

        for li in grad_layers:
            if li not in vecs:
                vecs[li] = np.zeros((MODEL_HIDDEN,), dtype=np.float32)

        return {
            "prompt": prompt,
            "completion": completion,
            "loss": float(out.loss.item()),
            "grad_vectors": vecs,
            "prompt_len": int(prompt_len),
        }
    finally:
        for h in hooks:
            h.remove()
        embeds.weight.requires_grad_(orig_req_grad)
        MODEL.zero_grad(set_to_none=True)
        free_memory()

def collect_representation_matrices(sample, rep_layers, variant=None):
    prompt, completion, enc, labels, prompt_len = build_teacher_forced_batch(sample, variant=variant)
    inputs = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.inference_mode():
        out = MODEL(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    attn_mask = inputs.get("attention_mask", None)
    mats = {}
    for li in rep_layers:
        h = out.hidden_states[li + 1]
        pooled = mean_pool_hidden(h, attention_mask=attn_mask)
        mats[li] = pooled.astype(np.float32)
    return {
        "prompt": prompt,
        "completion": completion,
        "rep_vectors": mats,
        "prompt_len": int(prompt_len),
    }

def greedy_eval(sample, variant=None):
    prompt = build_prompt(sample, variant=variant)
    full_output, completion = greedy_generate(prompt, max_new_tokens={"gsm8k": MAX_NEW_GSM8K, "strategyqa": MAX_NEW_STRATEGYQA, "mmlu": MAX_NEW_MMLU}[sample["task"]])
    if sample["task"] == "gsm8k":
        pred = parse_prediction("gsm8k", completion)
    elif sample["task"] == "strategyqa":
        pred = parse_prediction("strategyqa", completion)
    else:
        pred = parse_prediction("mmlu", completion, choices=sample.get("choices"))
    correct = exact_correct(sample["task"], pred, sample["gold"])
    return {
        "prompt": prompt,
        "full_output": full_output,
        "completion": completion,
        "prediction": pred,
        "correct": bool(correct),
    }

# ============================================================
# GATHER TASK DATA
# ============================================================

def load_task_samples():
    gsm = load_gsm8k_samples(GSM8K_N)
    sqa = load_strategyqa_samples(STRATEGYQA_N)
    mmlu = load_mmlu_samples(MMLU_SUBJECTS, MMLU_PER_SUBJECT)
    return {
        "gsm8k": gsm,
        "strategyqa": sqa,
        "mmlu": mmlu,
    }

# ============================================================
# ANALYSIS HELPERS
# ============================================================

def build_overall_task_vectors(grad_data, grad_layers):
    task_means = {}
    for task, samples in grad_data.items():
        per_layer = []
        for li in grad_layers:
            layer_vecs = [s["grad_vectors"][li] for s in samples]
            mean_vec = np.mean(np.stack(layer_vecs, axis=0), axis=0)
            per_layer.append(mean_vec)
        task_means[task] = np.concatenate(per_layer, axis=0)
    return task_means

def pcgrad_project(task_vectors, task_order=None):
    tasks = list(task_vectors.keys())
    if task_order is None:
        task_order = tasks[:]
    projected = {t: task_vectors[t].copy() for t in tasks}
    for ti in task_order:
        gi = projected[ti]
        for tj in tasks:
            if ti == tj:
                continue
            gj = projected[tj]
            dot = float(np.dot(gi, gj))
            if dot < 0:
                denom = float(np.dot(gj, gj) + 1e-12)
                gi = gi - (dot / denom) * gj
        projected[ti] = gi
    return projected

# ============================================================
# MAIN ANALYSIS PIPELINE
# ============================================================

def analyze():
    ensure_dir(OUT_DIR)
    ensure_dir(CSV_DIR)
    ensure_dir(PLOTS_DIR)
    ensure_dir(REPORTS_DIR)
    ensure_dir(CACHE_DIR)
    ensure_dir(NPZ_DIR)

    print("Loading model and tokenizer ...")
    load_model()

    all_layers = list(range(MODEL_NUM_LAYERS))
    grad_layers = list(range(max(0, MODEL_NUM_LAYERS - GRAD_LAYER_COUNT), MODEL_NUM_LAYERS))
    rep_layers = list(range(0, MODEL_NUM_LAYERS, REP_LAYER_STRIDE))

    print(f"Model layers: {MODEL_NUM_LAYERS}")
    print(f"Gradient layers: {grad_layers}")
    print(f"Representation layers: {rep_layers[:8]}{'...' if len(rep_layers) > 8 else ''}")

    samples_by_task = load_task_samples()
    all_eval_rows = []
    all_loss_rows = []
    all_grad_sample_rows = []
    all_rep_sample_rows = []
    all_grad_pair_rows = []
    all_rep_pair_rows = []
    all_rep_basis_rows = []

    grad_vectors_by_task = {}
    rep_vectors_by_task = {}

    for task, samples in samples_by_task.items():
        print(f"\n=== TASK: {task.upper()} ===")
        task_eval_rows = []
        task_loss_rows = []
        task_grad_samples = []
        task_rep_samples = []

        rep_matrix = np.zeros((len(samples), len(rep_layers), MODEL_HIDDEN), dtype=np.float32)
        grad_matrix = np.zeros((len(samples), len(grad_layers), MODEL_HIDDEN), dtype=np.float32)

        for i, sample in enumerate(samples):
            print(f"  sample {i+1}/{len(samples)} | {preview_text(sample['question'], 90)}")

            eval_out = greedy_eval(sample)
            task_eval_rows.append({
                "task": task,
                "sample_id": sample["sample_id"],
                "subject": sample.get("subject", ""),
                "source_index": sample["source_index"],
                "question": sample["question"],
                "choices": to_jsonable(sample.get("choices")),
                "gold": sample["gold"],
                "prompt": eval_out["prompt"],
                "completion": eval_out["completion"],
                "prediction": eval_out["prediction"],
                "correct": eval_out["correct"],
                "full_output": eval_out["full_output"],
                "prompt_tokens": len(TOKENIZER(eval_out["prompt"]).input_ids),
                "completion_tokens": len(TOKENIZER(eval_out["completion"]).input_ids),
            })

            loss_val, prompt, completion, enc, labels, prompt_len = task_loss(sample)
            task_loss_rows.append({
                "task": task,
                "sample_id": sample["sample_id"],
                "subject": sample.get("subject", ""),
                "source_index": sample["source_index"],
                "loss": loss_val,
                "prompt_len": prompt_len,
                "completion_len": len(TOKENIZER(completion).input_ids),
            })

            grad_info = collect_gradient_vectors(sample, grad_layers)
            gvecs = grad_info["grad_vectors"]
            for li_idx, li in enumerate(grad_layers):
                grad_matrix[i, li_idx] = gvecs[li]
                task_grad_samples.append({
                    "task": task,
                    "sample_id": sample["sample_id"],
                    "subject": sample.get("subject", ""),
                    "layer": li,
                    "grad_norm": float(np.linalg.norm(gvecs[li])),
                    "grad_vector": to_jsonable(gvecs[li].tolist()),
                })

            rep_info = collect_representation_matrices(sample, rep_layers)
            rvecs = rep_info["rep_vectors"]
            for lj_idx, lj in enumerate(rep_layers):
                rep_matrix[i, lj_idx] = rvecs[lj]
                task_rep_samples.append({
                    "task": task,
                    "sample_id": sample["sample_id"],
                    "subject": sample.get("subject", ""),
                    "layer": lj,
                    "rep_norm": float(np.linalg.norm(rvecs[lj])),
                    "rep_vector": to_jsonable(rvecs[lj].tolist()),
                })

            free_memory()

        eval_df = pd.DataFrame(task_eval_rows)
        loss_df = pd.DataFrame(task_loss_rows)
        grad_sample_df = pd.DataFrame(task_grad_samples)
        rep_sample_df = pd.DataFrame(task_rep_samples)

        save_df(eval_df, os.path.join(CSV_DIR, f"{task}_eval.csv"))
        save_df(loss_df, os.path.join(CSV_DIR, f"{task}_loss.csv"))
        save_df(grad_sample_df, os.path.join(CSV_DIR, f"{task}_grad_samples.csv"))
        save_df(rep_sample_df, os.path.join(CSV_DIR, f"{task}_rep_samples.csv"))

        np.savez_compressed(
            os.path.join(NPZ_DIR, f"{task}_grad_matrix.npz"),
            grad_matrix=grad_matrix.astype(np.float16),
            grad_layers=np.array(grad_layers, dtype=np.int32),
        )
        np.savez_compressed(
            os.path.join(NPZ_DIR, f"{task}_rep_matrix.npz"),
            rep_matrix=rep_matrix.astype(np.float16),
            rep_layers=np.array(rep_layers, dtype=np.int32),
        )

        grad_vectors_by_task[task] = [grad_matrix[i] for i in range(grad_matrix.shape[0])]
        rep_vectors_by_task[task] = [rep_matrix[i] for i in range(rep_matrix.shape[0])]

        task_summary = {
            "task": task,
            "n_samples": int(len(eval_df)),
            "greedy_accuracy": float(eval_df["correct"].mean()),
            "mean_loss": float(loss_df["loss"].mean()),
            "std_loss": float(loss_df["loss"].std()),
            "mean_completion_tokens": float(eval_df["completion_tokens"].mean()),
            "mean_prompt_tokens": float(eval_df["prompt_tokens"].mean()),
        }
        all_loss_rows.append(task_summary)
        save_json(task_summary, os.path.join(REPORTS_DIR, f"{task}_summary.json"))

        bar_plot(
            ["accuracy"], [task_summary["greedy_accuracy"]],
            f"{task.upper()} greedy accuracy", os.path.join(PLOTS_DIR, f"{task}_accuracy.png"), ylabel="Accuracy"
        )
        bar_plot(
            ["mean loss"], [task_summary["mean_loss"]],
            f"{task.upper()} mean teacher-forced loss", os.path.join(PLOTS_DIR, f"{task}_loss.png"), ylabel="Loss"
        )
        histogram(
            loss_df["loss"].tolist(),
            f"{task.upper()} loss distribution",
            os.path.join(PLOTS_DIR, f"{task}_loss_hist.png"),
            xlabel="Loss",
            bins=8,
        )

    task_mean_layer_vectors = {}
    task_mean_overall_vectors = {}
    task_mean_layer_norms = []
    pairwise_layer_cos_rows = []
    pairwise_layer_dot_rows = []
    sample_pair_rows = []

    for task in TASKS:
        sample_layer_mats = np.stack(grad_vectors_by_task[task], axis=0)
        task_mean_layer_vectors[task] = sample_layer_mats.mean(axis=0)
        task_mean_overall_vectors[task] = task_mean_layer_vectors[task].reshape(-1)

    task_pairs = list(combinations(TASKS, 2))
    gradient_conflict_by_layer = []
    for li_idx, layer_id in enumerate(grad_layers):
        pair_cos = {}
        pair_dot = {}
        for a, b in task_pairs:
            va = task_mean_layer_vectors[a][li_idx]
            vb = task_mean_layer_vectors[b][li_idx]
            cos_ = cosine(va, vb)
            dot_ = float(np.dot(va.reshape(-1), vb.reshape(-1)))
            pair_cos[f"{a}__{b}"] = cos_
            pair_dot[f"{a}__{b}"] = dot_
            pairwise_layer_cos_rows.append({
                "layer": layer_id,
                "task_a": a,
                "task_b": b,
                "cosine": cos_,
                "dot": dot_,
                "conflict": bool(dot_ < 0),
                "grad_norm_a": float(np.linalg.norm(va)),
                "grad_norm_b": float(np.linalg.norm(vb)),
            })
        conflict_rate = float(np.mean([1.0 if v < 0 else 0.0 for v in pair_dot.values()]))
        gradient_conflict_by_layer.append({
            "layer": layer_id,
            "conflict_rate": conflict_rate,
            **{f"cos_{k}": v for k, v in pair_cos.items()},
            **{f"dot_{k}": v for k, v in pair_dot.items()},
        })

    grad_conflict_df = pd.DataFrame(gradient_conflict_by_layer)
    grad_pair_df = pd.DataFrame(pairwise_layer_cos_rows)
    save_df(grad_conflict_df, os.path.join(CSV_DIR, "gradient_conflict_by_layer.csv"))
    save_df(grad_pair_df, os.path.join(CSV_DIR, "gradient_pairwise_layer_cosines.csv"))

    grad_overall_rows = []
    for a, b in task_pairs:
        va = task_mean_overall_vectors[a]
        vb = task_mean_overall_vectors[b]
        grad_overall_rows.append({
            "task_a": a,
            "task_b": b,
            "cosine": cosine(va, vb),
            "dot": float(np.dot(va, vb)),
            "norm_a": float(np.linalg.norm(va)),
            "norm_b": float(np.linalg.norm(vb)),
            "conflict": bool(np.dot(va, vb) < 0),
        })
    grad_overall_df = pd.DataFrame(grad_overall_rows)
    save_df(grad_overall_df, os.path.join(CSV_DIR, "gradient_overall_pairs.csv"))

    for layer_idx, layer_id in enumerate(grad_layers):
        for a, b in task_pairs:
            A = np.stack(grad_vectors_by_task[a], axis=0)[:, layer_idx, :]
            B = np.stack(grad_vectors_by_task[b], axis=0)[:, layer_idx, :]
            for ia in range(A.shape[0]):
                for ib in range(B.shape[0]):
                    sample_pair_rows.append({
                        "layer": layer_id,
                        "task_a": a,
                        "task_b": b,
                        "sample_a": ia,
                        "sample_b": ib,
                        "cosine": cosine(A[ia], B[ib]),
                        "dot": float(np.dot(A[ia], B[ib])),
                        "conflict": bool(np.dot(A[ia], B[ib]) < 0),
                    })
    sample_pair_df = pd.DataFrame(sample_pair_rows)
    save_df(sample_pair_df, os.path.join(CSV_DIR, "gradient_sample_pair_cosines.csv"))

    projected = pcgrad_project(task_mean_overall_vectors, task_order=TASKS)
    pcgrad_rows = []
    for task in TASKS:
        pcgrad_rows.append({
            "task": task,
            "orig_norm": float(np.linalg.norm(task_mean_overall_vectors[task])),
            "proj_norm": float(np.linalg.norm(projected[task])),
            "norm_change": float(np.linalg.norm(projected[task]) - np.linalg.norm(task_mean_overall_vectors[task])),
        })
    pcgrad_df = pd.DataFrame(pcgrad_rows)
    save_df(pcgrad_df, os.path.join(CSV_DIR, "pcgrad_projection.csv"))

    naive_combined = np.mean(np.stack([task_mean_overall_vectors[t] for t in TASKS], axis=0), axis=0)
    proj_combined = np.mean(np.stack([projected[t] for t in TASKS], axis=0), axis=0)
    pcgrad_summary = {
        "naive_combined_norm": float(np.linalg.norm(naive_combined)),
        "pcgrad_combined_norm": float(np.linalg.norm(proj_combined)),
        "naive_pairwise_conflicts": int((grad_overall_df["conflict"] == True).sum()),
        "pcgrad_task_order": TASKS,
        "pairwise_overall": grad_overall_rows,
    }
    save_json(pcgrad_summary, os.path.join(REPORTS_DIR, "pcgrad_summary.json"))

    rep_mats_by_task_layer = {t: {} for t in TASKS}
    for task in TASKS:
        rep_matrix = np.load(os.path.join(NPZ_DIR, f"{task}_rep_matrix.npz"))["rep_matrix"]
        for li_idx, layer_id in enumerate(rep_layers):
            rep_mats_by_task_layer[task][layer_id] = rep_matrix[:, li_idx, :].astype(np.float64)

    rep_pair_rows = []
    rep_basis_rows = []
    rep_centroid_rows = []
    rep_cka_rows = []
    for layer_id in rep_layers:
        bases = {}
        explained = {}
        centroids = {}
        for task in TASKS:
            X = rep_mats_by_task_layer[task][layer_id]
            centroids[task] = X.mean(axis=0)
            U, exp = top_pca_basis(X, k=min(SUBSPACE_K, max(1, X.shape[0] - 1), X.shape[1]))
            bases[task] = U
            explained[task] = exp
            rep_basis_rows.append({
                "layer": layer_id,
                "task": task,
                "n_samples": int(X.shape[0]),
                "hidden_dim": int(X.shape[1]),
                "pca_k": int(U.shape[1]),
                "explained_variance": float(exp),
                "centroid_norm": float(np.linalg.norm(centroids[task])),
            })
        for a, b in task_pairs:
            overlap = subspace_overlap(bases[a], bases[b])
            ccos = cosine(centroids[a], centroids[b])
            rep_pair_rows.append({
                "layer": layer_id,
                "task_a": a,
                "task_b": b,
                "subspace_overlap": overlap,
                "centroid_cosine": ccos,
                "explained_var_a": explained[a],
                "explained_var_b": explained[b],
            })
        for a, b in task_pairs:
            Xa = rep_mats_by_task_layer[a][layer_id]
            Xb = rep_mats_by_task_layer[b][layer_id]
            n = min(Xa.shape[0], Xb.shape[0])
            if n >= 2:
                Xa2 = Xa[:n]
                Xb2 = Xb[:n]
                rep_cka_rows.append({
                    "layer": layer_id,
                    "task_a": a,
                    "task_b": b,
                    "cka": linear_cka(Xa2, Xb2),
                    "n": int(n),
                })

    rep_pair_df = pd.DataFrame(rep_pair_rows)
    rep_basis_df = pd.DataFrame(rep_basis_rows)
    rep_cka_df = pd.DataFrame(rep_cka_rows)
    save_df(rep_pair_df, os.path.join(CSV_DIR, "representation_pairwise_overlap.csv"))
    save_df(rep_basis_df, os.path.join(CSV_DIR, "representation_pca_basis_stats.csv"))
    save_df(rep_cka_df, os.path.join(CSV_DIR, "representation_cka.csv"))

    eval_rows = []
    for task in TASKS:
        df = pd.read_csv(os.path.join(CSV_DIR, f"{task}_eval.csv"))
        loss_df = pd.read_csv(os.path.join(CSV_DIR, f"{task}_loss.csv"))
        eval_rows.append({
            "task": task,
            "n": int(len(df)),
            "greedy_accuracy": float(df["correct"].mean()),
            "mean_loss": float(loss_df["loss"].mean()),
            "std_loss": float(loss_df["loss"].std()),
            "mean_prompt_tokens": float(df["prompt_tokens"].mean()),
            "mean_completion_tokens": float(df["completion_tokens"].mean()),
        })
    eval_summary_df = pd.DataFrame(eval_rows)
    save_df(eval_summary_df, os.path.join(CSV_DIR, "evaluation_summary.csv"))

    bar_plot(eval_summary_df["task"].tolist(), eval_summary_df["greedy_accuracy"].tolist(), "Greedy accuracy by task", os.path.join(PLOTS_DIR, "task_accuracy.png"), ylabel="Accuracy")
    bar_plot(eval_summary_df["task"].tolist(), eval_summary_df["mean_loss"].tolist(), "Mean teacher-forced loss by task", os.path.join(PLOTS_DIR, "task_mean_loss.png"), ylabel="Loss")

    grad_cos_mat = np.eye(len(TASKS), dtype=np.float64)
    grad_dot_mat = np.eye(len(TASKS), dtype=np.float64)
    for i, a in enumerate(TASKS):
        for j, b in enumerate(TASKS):
            if i == j:
                continue
            grad_cos_mat[i, j] = cosine(task_mean_overall_vectors[a], task_mean_overall_vectors[b])
            grad_dot_mat[i, j] = float(np.dot(task_mean_overall_vectors[a], task_mean_overall_vectors[b]))
    heatmap(grad_cos_mat, TASKS, TASKS, "Overall gradient cosine similarity", os.path.join(PLOTS_DIR, "gradient_cosine_heatmap.png"), xlabel="Task", ylabel="Task", cmap="coolwarm")
    heatmap(grad_dot_mat, TASKS, TASKS, "Overall gradient dot products", os.path.join(PLOTS_DIR, "gradient_dot_heatmap.png"), xlabel="Task", ylabel="Task", cmap="coolwarm")

    for a, b in task_pairs:
        sdf = grad_pair_df[(grad_pair_df["task_a"] == a) & (grad_pair_df["task_b"] == b)].sort_values("layer")
        line_plot(sdf["layer"].tolist(), [sdf["cosine"].tolist()], [f"{a} vs {b}"], f"Gradient cosine by layer: {a} vs {b}", os.path.join(PLOTS_DIR, f"gradient_cosine_{a}_vs_{b}.png"), xlabel="Layer", ylabel="Cosine")
        line_plot(sdf["layer"].tolist(), [sdf["dot"].tolist()], [f"{a} vs {b}"], f"Gradient dot by layer: {a} vs {b}", os.path.join(PLOTS_DIR, f"gradient_dot_{a}_vs_{b}.png"), xlabel="Layer", ylabel="Dot product")

    line_plot(grad_conflict_df["layer"].tolist(), [grad_conflict_df["conflict_rate"].tolist()], ["conflict rate"], "Gradient conflict rate by layer", os.path.join(PLOTS_DIR, "gradient_conflict_rate.png"), xlabel="Layer", ylabel="Fraction of negative pairwise dots")
    histogram(grad_pair_df["cosine"].dropna().tolist(), "Distribution of pairwise gradient cosine values", os.path.join(PLOTS_DIR, "gradient_cosine_histogram.png"), xlabel="Cosine", bins=10)
    
    # Advanced Plot 1: Violin Distributions
    if not grad_pair_df.empty:
        violin_layer_plot(
            grad_pair_df, 
            "cosine", 
            "layer", 
            "Distribution of Pairwise Gradient Cosines Across Layers", 
            os.path.join(PLOTS_DIR, "gradient_cosine_violin.png"),
            ylabel="Cosine Similarity"
        )

    bar_plot(
        ["naive", "pcgrad"],
        [pcgrad_summary["naive_combined_norm"], pcgrad_summary["pcgrad_combined_norm"]],
        "Combined gradient norm: naive vs PCGrad",
        os.path.join(PLOTS_DIR, "pcgrad_combined_norm.png"),
        ylabel="Norm",
    )
    bar_plot(
        ["pairwise conflicts"],
        [pcgrad_summary["naive_pairwise_conflicts"]],
        "Number of negative task-gradient pairs",
        os.path.join(PLOTS_DIR, "pcgrad_conflicts.png"),
        ylabel="Count",
    )

    rep_overlap_mat = np.eye(len(TASKS), dtype=np.float64)
    rep_centroid_mat = np.eye(len(TASKS), dtype=np.float64)
    rep_cka_mat = np.eye(len(TASKS), dtype=np.float64)

    rep_overall_rows = []
    for i, a in enumerate(TASKS):
        for j, b in enumerate(TASKS):
            if i == j:
                continue
            overlaps = rep_pair_df[(rep_pair_df["task_a"] == a) & (rep_pair_df["task_b"] == b)]["subspace_overlap"].dropna().tolist()
            centroids = rep_pair_df[(rep_pair_df["task_a"] == a) & (rep_pair_df["task_b"] == b)]["centroid_cosine"].dropna().tolist()
            ckas = rep_cka_df[(rep_cka_df["task_a"] == a) & (rep_cka_df["task_b"] == b)]["cka"].dropna().tolist()
            rep_overlap_mat[i, j] = float(np.mean(overlaps)) if overlaps else float("nan")
            rep_centroid_mat[i, j] = float(np.mean(centroids)) if centroids else float("nan")
            rep_cka_mat[i, j] = float(np.mean(ckas)) if ckas else float("nan")
            rep_overall_rows.append({
                "task_a": a,
                "task_b": b,
                "mean_subspace_overlap": rep_overlap_mat[i, j],
                "mean_centroid_cosine": rep_centroid_mat[i, j],
                "mean_cka": rep_cka_mat[i, j],
            })
    rep_overall_df = pd.DataFrame(rep_overall_rows)
    save_df(rep_overall_df, os.path.join(CSV_DIR, "representation_overall_pairs.csv"))

    heatmap(rep_overlap_mat, TASKS, TASKS, "Mean representation subspace overlap", os.path.join(PLOTS_DIR, "rep_subspace_overlap_heatmap.png"), xlabel="Task", ylabel="Task", cmap="viridis")
    heatmap(rep_centroid_mat, TASKS, TASKS, "Mean representation centroid cosine", os.path.join(PLOTS_DIR, "rep_centroid_cosine_heatmap.png"), xlabel="Task", ylabel="Task", cmap="coolwarm")
    heatmap(rep_cka_mat, TASKS, TASKS, "Mean representation CKA", os.path.join(PLOTS_DIR, "rep_cka_heatmap.png"), xlabel="Task", ylabel="Task", cmap="coolwarm")

    for a, b in task_pairs:
        sdf = rep_pair_df[(rep_pair_df["task_a"] == a) & (rep_pair_df["task_b"] == b)].sort_values("layer")
        line_plot(sdf["layer"].tolist(), [sdf["subspace_overlap"].tolist(), sdf["centroid_cosine"].tolist()], ["subspace overlap", "centroid cosine"], f"Representation overlap by layer: {a} vs {b}", os.path.join(PLOTS_DIR, f"rep_overlap_{a}_vs_{b}.png"), xlabel="Layer", ylabel="Score")

        cka_sdf = rep_cka_df[(rep_cka_df["task_a"] == a) & (rep_cka_df["task_b"] == b)].sort_values("layer")
        line_plot(cka_sdf["layer"].tolist(), [cka_sdf["cka"].tolist()], ["CKA"], f"Representation CKA by layer: {a} vs {b}", os.path.join(PLOTS_DIR, f"rep_cka_{a}_vs_{b}.png"), xlabel="Layer", ylabel="CKA")

    merged_rows = []
    for layer_id in rep_layers:
        g_layer = grad_conflict_df[grad_conflict_df["layer"] == layer_id]
        r_layer = rep_pair_df[rep_pair_df["layer"] == layer_id]
        cka_layer = rep_cka_df[rep_cka_df["layer"] == layer_id]
        if not g_layer.empty and not r_layer.empty and not cka_layer.empty:
            merged_rows.append({
                "layer": layer_id,
                "gradient_conflict_rate": float(g_layer["conflict_rate"].mean()),
                "mean_rep_subspace_overlap": float(r_layer["subspace_overlap"].mean()),
                "mean_rep_centroid_cosine": float(r_layer["centroid_cosine"].mean()),
                "mean_cka": float(cka_layer["cka"].mean()),
            })
    
    merged_df = pd.DataFrame(merged_rows)
    save_df(merged_df, os.path.join(CSV_DIR, "layerwise_gradient_vs_representation.csv"))
    
    if len(merged_df) > 0:
        # Advanced Plot 2: Hexbin Density
        hexbin_plot(
            merged_df["gradient_conflict_rate"].tolist(),
            merged_df["mean_rep_subspace_overlap"].tolist(),
            "Layerwise Density: Grad Conflict vs Rep Overlap",
            os.path.join(PLOTS_DIR, "gradient_conflict_vs_rep_overlap_hexbin.png"),
            xlabel="Gradient conflict rate",
            ylabel="Mean subspace overlap"
        )
        
        # Advanced Plot 3: Metric Correlation Heatmap
        correlation_heatmap(
            merged_df,
            ["gradient_conflict_rate", "mean_rep_subspace_overlap", "mean_rep_centroid_cosine", "mean_cka"],
            "Cross-Metric Conflict Correlation",
            os.path.join(PLOTS_DIR, "layerwise_conflict_metric_correlation.png")
        )

    layer_norm_rows = []
    for task in TASKS:
        mat = np.stack(grad_vectors_by_task[task], axis=0)
        for li_idx, layer_id in enumerate(grad_layers):
            norms = np.linalg.norm(mat[:, li_idx, :], axis=1)
            layer_norm_rows.append({
                "task": task,
                "layer": layer_id,
                "mean_grad_norm": float(np.mean(norms)),
                "std_grad_norm": float(np.std(norms)),
            })
    layer_norm_df = pd.DataFrame(layer_norm_rows)
    save_df(layer_norm_df, os.path.join(CSV_DIR, "gradient_norms_by_layer.csv"))
    
    for task in TASKS:
        sdf = layer_norm_df[layer_norm_df["task"] == task].sort_values("layer")
        line_plot(sdf["layer"].tolist(), [sdf["mean_grad_norm"].tolist()], [task], f"Mean gradient norm by layer: {task}", os.path.join(PLOTS_DIR, f"grad_norm_{task}.png"), xlabel="Layer", ylabel="Grad norm")

    # Advanced Plot 4: Task Radar Chart
    radar_data = {}
    radar_categories = ["Greedy Acc", "Mean Loss (Inv)", "Prompt Len", "Grad Norm", "PCGrad Retention"]
    for task in TASKS:
        acc = float(eval_summary_df[eval_summary_df["task"] == task]["greedy_accuracy"].iloc[0])
        loss = float(eval_summary_df[eval_summary_df["task"] == task]["mean_loss"].iloc[0])
        inv_loss = max(0, 1.0 - (loss / max(1.0, float(eval_summary_df["mean_loss"].max())))) # Normalize loss
        p_len = float(eval_summary_df[eval_summary_df["task"] == task]["mean_prompt_tokens"].iloc[0]) / max(1.0, float(eval_summary_df["mean_prompt_tokens"].max()))
        grad_norm = float(layer_norm_df[layer_norm_df["task"] == task]["mean_grad_norm"].mean()) / max(1.0, float(layer_norm_df["mean_grad_norm"].max()))
        
        orig_norm = float(pcgrad_df[pcgrad_df["task"] == task]["orig_norm"].iloc[0])
        proj_norm = float(pcgrad_df[pcgrad_df["task"] == task]["proj_norm"].iloc[0])
        pcgrad_retention = proj_norm / max(1e-12, orig_norm)
        
        radar_data[task] = [acc, inv_loss, p_len, grad_norm, pcgrad_retention]

    if radar_data:
        radar_chart(
            radar_categories,
            radar_data,
            "Task-Level Multi-Dimensional Performance Fingerprint",
            os.path.join(PLOTS_DIR, "task_performance_radar.png")
        )

    summary = {
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "GSM8K_N": GSM8K_N,
            "STRATEGYQA_N": STRATEGYQA_N,
            "MMLU_PER_SUBJECT": MMLU_PER_SUBJECT,
            "MMLU_SUBJECTS": MMLU_SUBJECTS,
            "GRAD_LAYER_COUNT": GRAD_LAYER_COUNT,
            "REP_LAYER_STRIDE": REP_LAYER_STRIDE,
            "SUBSPACE_K": SUBSPACE_K,
            "USE_MMLU_DELIMITER": USE_MMLU_DELIMITER,
            "USE_GSM8K_REASONING_PROMPT": USE_GSM8K_REASONING_PROMPT,
            "USE_STRATEGYQA_REASONING_PROMPT": USE_STRATEGYQA_REASONING_PROMPT,
        },
        "evaluation_summary": eval_rows,
        "gradient_overall_pairs": grad_overall_rows,
        "pcgrad": pcgrad_summary,
        "representation_overall_pairs": rep_overall_rows,
    }
    save_json(summary, os.path.join(REPORTS_DIR, "summary.json"))

    md = ["# Task interference / gradient conflict report\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Tasks: {', '.join(TASKS)}")
    md.append(f"- GSM8K samples: {GSM8K_N}")
    md.append(f"- StrategyQA samples: {STRATEGYQA_N}")
    md.append(f"- MMLU samples per subject: {MMLU_PER_SUBJECT}")
    md.append(f"- MMLU subjects: {', '.join(MMLU_SUBJECTS)}")
    md.append(f"- Gradient layers: {grad_layers}")
    md.append(f"- Representation layers: {rep_layers[:10]}{'...' if len(rep_layers) > 10 else ''}")
    md.append("\n## Evaluation summary\n")
    md.append("| Task | Greedy accuracy | Mean loss | Std loss | Mean prompt tokens | Mean completion tokens |")
    md.append("|---|---:|---:|---:|---:|---:|")
    for _, r in eval_summary_df.iterrows():
        md.append(f"| {r['task']} | {r['greedy_accuracy']:.3f} | {r['mean_loss']:.3f} | {r['std_loss']:.3f} | {r['mean_prompt_tokens']:.1f} | {r['mean_completion_tokens']:.1f} |")

    md.append("\n## Gradient conflict summary\n")
    md.append("| Task A | Task B | Cosine | Dot | Conflict |")
    md.append("|---|---|---:|---:|---|")
    for _, r in grad_overall_df.iterrows():
        md.append(f"| {r['task_a']} | {r['task_b']} | {r['cosine']:.4f} | {r['dot']:.4e} | {str(bool(r['conflict']))} |")

    md.append("\n## PCGrad simulation\n")
    md.append(f"- Naive combined norm: {pcgrad_summary['naive_combined_norm']:.4f}")
    md.append(f"- PCGrad combined norm: {pcgrad_summary['pcgrad_combined_norm']:.4f}")
    md.append(f"- Pairwise negative gradient pairs: {pcgrad_summary['naive_pairwise_conflicts']}")

    md.append("\n## Representation overlap summary\n")
    md.append("| Task A | Task B | Mean subspace overlap | Mean centroid cosine | Mean CKA |")
    md.append("|---|---|---:|---:|---:|")
    for _, r in rep_overall_df.iterrows():
        md.append(f"| {r['task_a']} | {r['task_b']} | {r['mean_subspace_overlap']:.4f} | {r['mean_centroid_cosine']:.4f} | {r['mean_cka']:.4f} |")

    md.append("\n## Why this matters\n")
    md.append("- Negative task-gradient cosine suggests cross-task interference in shared layers.")
    md.append("- Lower representation overlap indicates the tasks occupy different subspaces, supporting routing / adapters.")
    md.append("- PCGrad reduces direct gradient conflict without changing the model.")
    md.append("- The MMLU delimiter run can be compared against the plain baseline to see whether formatting changes the interference pattern.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    save_df(eval_summary_df, os.path.join(CSV_DIR, "evaluation_summary.csv"))
    save_df(grad_overall_df, os.path.join(CSV_DIR, "gradient_overall_pairs.csv"))
    save_df(grad_conflict_df, os.path.join(CSV_DIR, "gradient_conflict_by_layer.csv"))
    save_df(grad_pair_df, os.path.join(CSV_DIR, "gradient_pairwise_layer_cosines.csv"))
    save_df(sample_pair_df, os.path.join(CSV_DIR, "gradient_sample_pair_cosines.csv"))
    save_df(pcgrad_df, os.path.join(CSV_DIR, "pcgrad_projection.csv"))
    save_df(rep_pair_df, os.path.join(CSV_DIR, "representation_pairwise_overlap.csv"))
    save_df(rep_basis_df, os.path.join(CSV_DIR, "representation_pca_basis_stats.csv"))
    save_df(rep_cka_df, os.path.join(CSV_DIR, "representation_cka.csv"))
    save_df(rep_overall_df, os.path.join(CSV_DIR, "representation_overall_pairs.csv"))
    save_df(layer_norm_df, os.path.join(CSV_DIR, "gradient_norms_by_layer.csv"))
    if len(merged_df) > 0:
        save_df(merged_df, os.path.join(CSV_DIR, "layerwise_gradient_vs_representation.csv"))

    if RUN_MMLU_PLAIN_BASELINE:
        plain_samples = samples_by_task["mmlu"]
        plain_eval_rows = []
        for sample in plain_samples:
            prompt = build_prompt(sample, variant="plain")
            full_out, completion = greedy_generate(prompt, max_new_tokens=MAX_NEW_MMLU)
            pred = extract_mcq(completion, choices=sample["choices"])
            plain_eval_rows.append({
                "sample_id": sample["sample_id"],
                "subject": sample.get("subject", ""),
                "question": sample["question"],
                "gold": sample["gold"],
                "prediction": pred,
                "correct": bool(pred == sample["gold"]),
                "full_output": full_out,
            })
        plain_eval_df = pd.DataFrame(plain_eval_rows)
        save_df(plain_eval_df, os.path.join(CSV_DIR, "mmlu_plain_baseline_eval.csv"))
        if not plain_eval_df.empty:
            bar_plot(
                ["delimiter", "plain"],
                [eval_summary_df[eval_summary_df["task"] == "mmlu"]["greedy_accuracy"].iloc[0], float(plain_eval_df["correct"].mean())],
                "MMLU greedy accuracy: delimiter vs plain",
                os.path.join(PLOTS_DIR, "mmlu_delimiter_vs_plain_accuracy.png"),
                ylabel="Accuracy",
            )

    print("\n===== FINAL SUMMARY =====")
    if not eval_summary_df.empty:
        print(eval_summary_df.to_string(index=False))
    print("\nGradient pairs:")
    if not grad_overall_df.empty:
        print(grad_overall_df.to_string(index=False))
    print("\nPCGrad:")
    print(json.dumps(pcgrad_summary, indent=2))
    print("\nRepresentation pairs:")
    if not rep_overall_df.empty:
        print(rep_overall_df.to_string(index=False))
    print(f"\nSaved outputs to: {OUT_DIR}")

    free_memory(MODEL)

# ============================================================
# ENTRY
# ============================================================

if __name__ == "__main__":
    analyze()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Generating validation split:   0%|          | 0/18 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]


=== TASK: GSM8K ===
  sample 1/5 | Carol and Jennifer are sisters from Los Angeles who love collecting signatures from celebr...


/tmp/ipykernel_23/3140571732.py:340: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if float(denom) <= 0:


  sample 2/5 | A team of 4 painters worked on a mansion for 3/8ths of a day every day for 3 weeks. How ma...
  sample 3/5 | It costs $194 per meter to repave a street. Monica's street is 150 meters long. How much m...
  sample 4/5 | Richard lives in an apartment building with 15 floors. Each floor contains 8 units, and 3/...
  sample 5/5 | An ice cream truck is traveling through a neighborhood. Children from various homes have s...

=== TASK: STRATEGYQA ===
  sample 1/5 | Could boolean algebra be described as binary?
  sample 2/5 | Would lumberjacks get full after eating three dosa?
  sample 3/5 | Would a member of the United States Air Force get a discount at Dunkin Donuts?
  sample 4/5 | In a hypothetical race between a Swallow and an American Woodcock, would the Swallow win?
  sample 5/5 | Were number of states in Ancient Greece underwhelming compared to US states in 1900?

=== TASK: MMLU ===
  sample 1/9 | Statement 1 | If a group has an element of order 10, then it has elements of